In [1]:
import os
import numpy as np
import xarray as xr

In [2]:
# === Processing function ===
def calculate_6m_temp_mean(temp_data, month_array):
    years = month_array.year.values

    T6M = xr.DataArray(  # 6 monthly average temperature
            np.nan,
            dims=["year", "lat", "lon"],
            coords={"year": month_array.year, "lat": month_array.lat, "lon": month_array.lon},
        )

    for y in years:
        print(f"Processing year {y}")
        for i in temp_data.lat:
            for j in temp_data.lon:
                # Extract the starting month as an integer
                start_month = int(month_array.sel(year=y, lat=i, lon=j).values)

                # Handle month wrap-around across years
                if start_month + 5 > 12:
                    end_month = (start_month + 5) % 12
                    end_year = y + 1
                else:
                    end_month = start_month + 5
                    end_year = y

                # Slice and calculate 6-month mean
                temp_6m = temp_data.sel(
                    time=slice(f"{y}-{start_month:02d}", f"{end_year}-{end_month:02d}"),
                    lat=i, lon=j
                ).mean()

                T6M.loc[dict(year=y, lat=i, lon=j)] = temp_6m
    return T6M

In [ ]:
# === Path config ===
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/OSDMA8_March/"
TREFHT_DIR = "/glade/work/awells/air_quality/CESM/TREFHT/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/T6M/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(9, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        temp = xr.open_dataarray(f"{TREFHT_DIR}TREFHT_CESM2_{scenario}_{ens_num:02d}_203501-206912.nc")
        ozone_month = xr.open_dataarray(f"{OSDMA8_DIR}OSDMA8_startmonth_CESM2_{scenario}_{ens_num:02d}_20350101-20691231.nc")

        T6M = calculate_6m_temp_mean(temp, ozone_month)

        out_file = f"T6M_CESM2_{scenario}_{ens_num:02d}_203501-206912.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        T6M.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 09
Processing year 2035
Processing year 2036
Processing year 2037
Processing year 2038
Processing year 2039
Processing year 2040
Processing year 2041
Processing year 2042
Processing year 2043
Processing year 2044
Processing year 2045
Processing year 2046
Processing year 2047
Processing year 2048
Processing year 2049
Processing year 2050
Processing year 2051
Processing year 2052
